In [1]:
import os
import celldega as dega

sample = 'Xenium_V1_human_Pancreas_FFPE_outs'
DATA_DIR = '../data'
tile_size = 250
path_landscape_files = f'{sample}'

data_dir = f'{DATA_DIR}/{sample}'

for folder in [data_dir, path_landscape_files]:
    if not os.path.exists(folder):
        os.mkdir(folder)
        print (folder)

if os.path.exists(f'{data_dir}/experiment.xenium'):
    technology = 'Xenium'

In [2]:
# Unzip compressed files in Xenium data folder
dega.pre._xenium_unzipper(data_dir)

# Check required files for preprocessing
dega.pre._check_required_files(technology, data_dir)

# Calculate and save CBG gene parquet files
cbg = dega.pre.read_cbg_mtx(f"{data_dir}/cell_feature_matrix")
dega.pre.save_cbg_gene_parquets(path_landscape_files, cbg, verbose=True)

# Make meta gene files
path_cbg = f"{data_dir}/cell_feature_matrix"
path_output = f"{path_landscape_files}/meta_gene.parquet" # or should it be 'gene_metadata.parquet'. TODO test it on full dataset before merging
dega.pre.make_meta_gene(technology, path_cbg, path_output)

# Create image tiles
dega.pre.create_image_tiles(technology, data_dir, path_landscape_files, image_tile_layer='all')

# Write transform_file
transformation_matrix = dega.pre.write_xenium_transform(data_dir, path_landscape_files)

# Make cell image coord
path_transformation_matrix = f'{path_landscape_files}/xenium_transform.csv'
path_meta_cell_micron = f'{data_dir}/cells.csv.gz'
path_meta_cell_image = f'{path_landscape_files}/cell_metadata.parquet'

dega.pre.make_meta_cell_image_coord(
    technology, 
    path_transformation_matrix, 
    path_meta_cell_micron, 
    path_meta_cell_image, 
    image_scale=1
)

# Create cluster and meta cluster files
clusters = dega.pre.create_cluster_and_meta_cluster(technology, data_dir, path_landscape_files)

# Generating transcript tiles
path_trx = f'{data_dir}/transcripts.parquet'
path_trx_tiles = f'{path_landscape_files}/transcript_tiles'
print("\n========Generating transcript tiles========")
print (f'Generating transcript tiles for {data_dir} and saving at {path_trx_tiles}')


tile_bounds = dega.pre.make_trx_tiles(
        technology,
        path_trx,
        path_transformation_matrix,
        path_trx_tiles,
        coarse_tile_factor=10,
        tile_size=tile_size,
        chunk_size=100000,
        verbose=False,
        image_scale=1,
        max_workers=2)

# Generating boundary tiles
path_cell_boundaries = f'{data_dir}/cell_boundaries.parquet'
path_output = f'{path_landscape_files}/cell_segmentation'
print (f'Generating transcript tiles for {data_dir} and saving at {path_output}')


print("\n========Generating boundary tiles========")
cells_orig = dega.pre.make_cell_boundary_tiles(
    technology,
    path_cell_boundaries,
    path_meta_cell_micron,
    path_transformation_matrix,
    path_output,
    coarse_tile_factor=10,
    tile_size=tile_size,
    tile_bounds=tile_bounds,
    image_scale=1,
    max_workers=2)

# Create cluster based gene expression
df_sig = dega.pre.cluster_gene_expression(technology, data_dir, path_landscape_files, cbg)

# Save landscape parameters
dega.pre.save_landscape_parameters(
    technology, 
    path_landscape_files,
    'dapi_files',
    tile_size=tile_size,
    image_info=dega.pre.get_image_info(technology, 'all'),
    image_format='.webp'
)

print("Preprocessing completed successfully.")


========Unzip and extract Xenium-related files========
cells.csv already exists. Skipping decompression.
cells.zarr directory already exists. Skipping unzipping.
analysis directory already exists. Skipping extraction.
cell_feature_matrix directory already exists. Skipping extraction.
All files have been successfully extracted or skipped.
Restored working directory to '/Users/whuan/dev/celldega/notebooks'.

========Check if all required files or directories exist========
All required files or directories for technology 'Xenium' are present in '../data/Xenium_V1_human_Pancreas_FFPE_outs'.

========Save cbg gene parquet========
Processing gene 0: ABCC11
Processing gene 100: CLECL1
Processing gene 200: IL1RL1
Processing gene 300: RGS16
Processing gene 400: NegControlCodeword_0503
Processing gene 500: UnassignedCodeword_0459

========Write meta gene files========
cbg is a dense DataFrame. Proceeding with dense operations.
Calculating mean expression
Calculating variance
All meta gene files

Processing chunks: 100%|██████████| 81/81 [00:00<00:00, 277.67it/s]
Processing coarse tiles: 84tile [01:50,  1.32s/tile]


Generating transcript tiles for ../data/Xenium_V1_human_Pancreas_FFPE_outs and saving at Xenium_V1_human_Pancreas_FFPE_outs/cell_segmentation

========Generating boundary tiles========


Processing coarse tiles: 100%|██████████| 14/14 [00:03<00:00,  3.64it/s]



========Create cluster gene expression (df_sig)========
Cluster-specific gene expression signatures saved successfully.

========Save landscape parameters========
Done.
Preprocessing completed successfully.


/Users/whuan/opt/anaconda3/envs/celldega_env_2025/lib/python3.10/site-packages/pandas/io/parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


In [ ]:

# To run the whole thing in command line:
# Git clone celldega repo and cd to celldega, then

"""
python run_pre_processing.py \
    --sample Xenium_V1_human_Pancreas_FFPE_outs \
    --data_root_dir data \
    --tile_size 250 \
    --image_tile_layer 'all' \
    --path_landscape_files notebooks/Xenium_V1_human_Pancreas_FFPE_outs
"""